# Overview

This notebook is designed for the CatBoost model training.
We will use gradient boosting with extracted images embeddings in order to predict the litotypes on the source images.

Unfortunately, CatBoost doesn't natively support MPS, so CPU calculations will be used instead. 

## 1. Imports and Settings

In [1]:
import ast
import datetime
import os
import warnings

import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [ ]:
BASE_PATH = "/Users/locked15/Repos.localized/Python/Study/UUST/Course 3/Semester 2/Practice/sludge-utilities/apps/gb/gb-training_lab/"

DATASET_PATH = "data/interim/"
DATASET_TARGET_FILE = "metadata_dinov3_embeddings.parquet"

FEATURE_COLS = [
    "interval_start", "interval_end"
]
TARGET_COLS = [
    'sandstone_sludge', 'siltstone_sludge', 'argillite_sludge'
]

OUTPUT_TYPE = "catboost"
OUTPUT_MODELS_PATH = "output/models/"
OUTPUT_METRICS_PATH = "output/metrics/"
MODEL_NAME = "catboost_model-{0}.cbm"
METRICS_NAME = "catboost-model-{0}-{1}.csv"

In [3]:
warnings.filterwarnings("ignore")

np.random.seed(42)

## 2. Dataset Loading

In [ ]:
df = pd.read_parquet(os.path.join(BASE_PATH, DATASET_PATH, DATASET_TARGET_FILE))


print("Dataset loaded. Sample: ")
print(df.head())

Dataset loaded. Sample: 
   well_id  device_no  interval_start  interval_end  sandstone_sludge  \
0        4        1.0            2675          2680                 5   
1        4        2.0            2680          2685                 5   
2        4        3.0            2685          2690                 5   
3        4        4.0            2690          2695                 5   
4        4        5.0            2695          2700                 5   

   siltstone_sludge  argillite_sludge  radiolarite_sludge  coal_sludge  \
0                25                70                   0            0   
1                25                70                   0            0   
2                30                65                   0            0   
3                30                65                   0            0   
4                25                70                   0            0   

   limestone_sludge  ...  oil_saturation  calcite_carbonatometry  \
0                 0  ..

## 3. Embeddings Processing

In [5]:
def str_to_array(x):
    if isinstance(x, str):
        try:
            return np.array(ast.literal_eval(x))
        except:
            return np.array(x)
    return x

df['lba_dinov3_emb'] = df['lba_dinov3_emb'].apply(str_to_array)
df['sludge_dinov3_emb'] = df['sludge_dinov3_emb'].apply(str_to_array)

print("Embeddings formatted as arrays.")
print(df['sludge_dinov3_emb'].values[0].shape)

Embeddings formatted as arrays.
(1024,)


## 4. Features and Targets Preparation

In [6]:
lba_emb_df = pd.DataFrame(
    df['lba_dinov3_emb'].tolist(),
    columns = [f"lba_emb_{i}" \
               for i in range(df["lba_dinov3_emb"].values[0] \
                                                  .shape[0])
              ]
)
sludge_emb_df = pd.DataFrame(
    df['sludge_dinov3_emb'].tolist(),
    columns = [f"sludge_emb_{i}" \
               for i in range(df["sludge_dinov3_emb"].values[0] \
                                                     .shape[0])
              ]
)

input_df = pd.concat([df[FEATURE_COLS].reset_index(drop = True), sludge_emb_df, lba_emb_df], axis = 1)
print("Input features are prepared.")
print(input_df.head())

output_df = df[TARGET_COLS].reset_index(drop = True)
print("Target variables are prepared.")
print(output_df.head())


Input features are prepared.
   interval_start  interval_end  sludge_emb_0  sludge_emb_1  sludge_emb_2  \
0            2675          2680     -0.444155     -0.257195      0.362337   
1            2680          2685     -0.134183     -0.076541     -0.063532   
2            2685          2690     -0.057407     -0.184069     -0.064041   
3            2690          2695     -0.082111     -0.067013     -0.096281   
4            2695          2700     -0.271062      0.104002     -0.005017   

   sludge_emb_3  sludge_emb_4  sludge_emb_5  sludge_emb_6  sludge_emb_7  ...  \
0      0.547643     -0.477527      0.593543      0.217066     -0.292527  ...   
1      0.363979     -0.575249      0.321932      0.319753      0.165333  ...   
2      0.224925     -0.339440      0.409055      0.112859      0.032745  ...   
3      0.436976     -0.512890      0.217838      0.152675      0.536699  ...   
4      0.572512     -0.429888      0.196399      0.251413      0.272011  ...   

   lba_emb_1014  lba_emb_10

## 5. Training

### 5.1. Preparation

In [ ]:
params = {
    'iterations': 2500,
    'learning_rate': 0.05,
    'depth': 8,
    'loss_function': 'MultiRMSE',
    'eval_metric': 'MultiRMSE',
    'random_seed': 42,
    'early_stopping_rounds': 150,
    'verbose': 100,
    'task_type': 'CPU',
    'devices': '0'
}

In [19]:
class TabularDataset:
    def __init__(self, X, y, groups = None):
        self.X = X
        self.y = y
        self.groups = groups

    def get_fold(self, train_idx, val_idx):
        X_train = self.X.iloc[train_idx]
        X_val = self.X.iloc[val_idx]

        y_train = self.y.iloc[train_idx]
        y_val = self.y.iloc[val_idx]

        return X_train, X_val, y_train, y_val

    def extract_folds(self, n_splits = 4):
        gkf = GroupKFold(n_splits = n_splits)
        folds = []
        for train_idx, val_idx in gkf.split(self.X, self.y, self.groups):
            X_train = self.X.iloc[train_idx]
            X_val = self.X.iloc[val_idx]
            y_train = self.y.iloc[train_idx]
            y_val = self.y.iloc[val_idx]
            folds.append((X_train, X_val, y_train, y_val))
        return folds


In [ ]:
def normalize_predictions(predictions):
    predictions_sum = predictions.sum(axis = 1, keepdims = True)
    return predictions / predictions_sum * 100

def train_model(X_train, y_train, X_val, y_val, params):
    model = CatBoostRegressor(**params)
    model.fit(
        X_train,
        y_train,
        eval_set = (X_val, y_val),
        use_best_model = True,
        verbose = True
    )
    return model

def get_predictions_by_model(model, X_val, y_val):
    pred = model.predict(X_val)
    pred = normalize_predictions(pred)
    pred_df = pd.DataFrame(
        pred,
        columns = y_val.columns,
        index = y_val.index
    )
    return pred_df

def calculate_metrics(
    y_true: pd.DataFrame,
    y_predicted: pd.DataFrame
) -> dict:
    metrics = {}
    for target in y_true.columns:
        metrics[target] = {
            "mae": mean_absolute_error(
                y_true[target],
                y_predicted[target]
            ),
            "rmse": root_mean_squared_error(
                y_true[target],
                y_predicted[target]
            ),
            "r2": r2_score(
                y_true[target],
                y_predicted[target]
            )
        }
    return metrics

def print_metrics(metrics):
    print("!== Cross-Validation Metrics ==!")
    for target in metrics:
        print(target)
        # MAE:
        print(
            f"MAE: {np.mean(metrics[target]['mae']):.4f} "
            f"+/- {np.std(metrics[target]['mae']):.4f}"
        )
        # RMSE:
        print(
            f"RMSE: {np.mean(metrics[target]['rmse']):.4f} "
            f"+/- {np.std(metrics[target]['rmse']):.4f}"
        )
        # R2:
        print(
            f"R2: {np.mean(metrics[target]['r2']):.4f} "
            f"+/- {np.std(metrics[target]['r2']):.4f}"
        )
        print("===")


In [ ]:
def run_cv(
    input_df,
    output_df,
    groups,
    params,
    n_splits = 4
):
    dataset = TabularDataset(input_df, output_df, groups)
    folds = dataset.extract_folds(n_splits = n_splits)
    
    aggregated_metrics = {
        target: {
            "mae": [],
            "rmse": [],
            "r2": []
        }
        for target in output_df.columns
    }
    
    # Ensure directory for metrics exists
    os.makedirs(os.path.join(BASE_PATH, OUTPUT_METRICS_PATH, OUTPUT_TYPE), exist_ok = True)
    
    for fold_idx, (X_train, X_val, y_train, y_val) in enumerate(folds):
        model = train_model(X_train, y_train, X_val, y_val, params)
        predictions = get_predictions_by_model(model, X_val, y_val)
        
        # Save predictions to csv with original values side-by-side
        pred_renamed = predictions.rename(columns = lambda x: f"{x}_pred")
        y_val_renamed = y_val.rename(columns = lambda x: f"{x}_true")
        combined_df = pd.concat([pred_renamed, y_val_renamed], axis = 1)
        
        current_datetime = datetime.datetime.now().strftime("%Y-%m-%d_%H:%M:%S")
        filename = METRICS_NAME.format(fold_idx + 1, current_datetime)
        filepath = os.path.join(BASE_PATH, OUTPUT_METRICS_PATH, OUTPUT_TYPE, filename)
        combined_df.to_csv(filepath)
        
        fold_metrics = calculate_metrics(y_val, predictions)
        
        for target in fold_metrics:
            for metric_name in fold_metrics[target]:
                aggregated_metrics[target][metric_name].append(
                    fold_metrics[target][metric_name]
                )
                
        print(f"Fold {fold_idx + 1} completed.")
        
    return aggregated_metrics

### 6.2. Training

In [ ]:
groups = df["well_id"]
metrics = run_cv(
    input_df,
    output_df,
    groups,
    params,
    n_splits = 4
)
print_metrics(metrics)

Fold 1 completed.
Fold 2 completed.
Fold 3 completed.

=== Cross-validation metrics ===

sandstone_sludge
  MAE : 25.9043 ± 8.3028
  RMSE: 28.6620 ± 8.1025
  R²  : -6.5015 ± 7.4255

siltstone_sludge
  MAE : 27.9326 ± 10.5833
  RMSE: 29.7711 ± 9.9618
  R²  : -6.5362 ± 4.6710

argillite_sludge
  MAE : 43.6433 ± 16.8868
  RMSE: 45.9952 ± 16.2465
  R²  : -15.3497 ± 10.8404

